In [ ]:
from sklearn import  linear_model
from sklearn.preprocessing import OneHotEncoder
import statsmodels.formula.api as smf 
import statsmodels.api as sm 
import seaborn as sns 
from ISLP import load_data, confusion_table
from ISLP.models import contrast
from sklearn. discriminant_analysis import \
(LinearDiscriminantAnalysis as LDA,
QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from ISLP.models import (ModelSpec as MS, summarize ) 
from sklearn.metrics import confusion_matrix

#### Scratch for chapter

In [ ]:
bikers_df = load_data("Bikeshare")
bikers_df = bikers_df.assign(log_bikers=np.log(bikers_df["bikers"]))

bikers_df["weathersit"].unique()
sns.scatterplot(data=bikers_df, x="hr", y="bikers")
sns.scatterplot(data=bikers_df, x="hr", y="log_bikers")

In [ ]:
# let me give logistic regresssion a go 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 

bikers_poisson = bikers_df[["hr", "workingday", "weathersit", "temp", "bikers"]]
categorical_features = ["weathersit"] 
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop="first"), categorical_features), 
    ],
    remainder='passthrough'
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('regressor', linear_model.PoissonRegressor())
]) 

model.fit(bikers_poisson[["hr", "weathersit", "temp"]], bikers_poisson["bikers"])

In [ ]:
model = smf.poisson("bikers ~ weathersit + workingday + temp", data=bikers_poisson).fit()
model.summary()

In [ ]:
# Lab
smarket_df = load_data('Smarket') 
numerical_cols = [col for col in smarket_df.columns if col != "Direction"] 
smarket_df[numerical_cols].corr() 

In [ ]:
allvars = smarket_df.columns.drop(["Today", "Direction", "Year"]) 
design = MS(allvars) 
X = design.fit_transform(smarket_df) 

y = smarket_df.Direction == 'Up'
glm = sm.GLM(y, X, family=sm.families.Binomial()) 
results = glm.fit() 

summarize(results) 



#### Applied Exercises 

#### Ex 12

In [ ]:
# (a) Just a cross correl 
weekly = load_data("Weekly") 
weekly = weekly.assign(Direction=weekly["Direction"].map({"Down": 0, "Up" : 1}))

weekly.head() 
sns.pairplot(weekly)

In [ ]:
# just a stab on how this would be done but also using ISLP MS ( I am not sure why ?) 
X = MS([f"Lag{ii}" for ii in range(1, 6)] + ["Volume"]).fit_transform(weekly) 

y = weekly["Direction"] 
logit = LogisticRegression(C=1e10, solver='liblinear').fit(X, y) 

logit_insample_predict = logit.predict_proba(X) 

# but it is easier to get a summary with statsmodels  

sm_logit = sm.Logit(y, X).fit()

sm_logit.summary()

In [ ]:
# Ex 16 - The Boston data 
from utils import confusion_table, total_error_rate
boston = load_data("Boston") 

boston["crime_up"] = (boston["crim"] > boston["crim"].median()).astype(int)
boston.info()

In [ ]:
# Look at this nice trick to change the graph types in seaborn between diagonal, upper triangular and 
# lower triangular 
g = sns.PairGrid(boston)
g.map_upper(plt.scatter, s=3, alpha=0.7)
g.map_diag(plt.hist, alpha=0.6)

def lower_plots(x, y, **kwargs):
    if y.name in ['crim01', 'chas']: 
        num_colors_needed = y.nunique()
        sns.boxplot(x=x, y=y, orient='h', hue=y, palette=sns.color_palette('Blues_d', n_colors=num_colors_needed), **kwargs)  # Horizontal boxplot
    else:
        sns.kdeplot(x=x, y=y, cmap='Blues_d', **kwargs)  # KDE plot

g.map_lower(lower_plots)
g.figure.set_size_inches(14, 14);


In [ ]:
# mandatory coloured correls 
fig, ax = plt.subplots(1, 1, figsize=(14, 14))

palette = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(boston.corr(), ax= ax, annot=True, cmap=palette);


In [ ]:
def fit_and_get_error_rate(model, X, y, 
                           test_size: float = 0.3, 
                           random_state: np.random.RandomState | int = 1):
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                        test_size=test_size, 
                                                        random_state=random_state)
    
    mdl = model
    
    mdl.fit(X_train, y_train)
    pred = mdl.predict(X_test) 
    conf_mat = confusion_table(pred, y_test)
    return total_error_rate(conf_mat)

In [ ]:
X = boston[['indus', 'nox', 'age', 'dis', 'rad', 'tax']]
y = boston['crime_up']

for model in [LDA(), QDA(), LogisticRegression(max_iter=1000), GaussianNB()]:
    print(f'{model.__str__().split("(")[0]}: {fit_and_get_error_rate(model, X, y)}\n')